## Основные алгоритмы симметричного шифрования

### Материалы курса


### Цель работы
Ознакомиться с принципами реализации простых алгоритмов шифрования и дешифрования текста на примере учебных моделей.

### Содержание работы
- Реализация обобщённого шифра Цезаря (сдвиг на целое число) для символов Unicode с модулем 65536.
- Реализация шифра Виженера (циклический строковый ключ) в той же Unicode-модели.
- Восстановление открытого текста без ключа методом частотной эвристики (для Цезаря).
- Шифр Вернама (XOR): повторяющийся ключ и схема одноразового блокнота (OTP).
- По желанию: CBC-подобная цепочка блоков для XOR.

### Примечание
Все схемы в ноутбуке имеют **учебный** характер: они иллюстрируют идеи криптографии и не предназначены для защиты реальных данных в прикладных системах.


### Работа с ноутбуком
- Выполняйте ячейки **последовательно сверху вниз**.
- В ячейках с пометкой TODO допишите код и снова выполните ячейку.
- После каждой реализованной функции запускайте ближайшую проверочную ячейку и сравнивайте вывод с ожидаемым поведением.


## Подготовка: кодировка символов, функции ord и chr

Символы задаются кодами Unicode. В Python:
- ord(c) — целочисленный код символа c (кодовая точка);
- chr(i) — символ с кодом i.

В данной работе преобразования выполняются по модулю 65536. Это соответствует диапазону BMP (первые 65536 кодовых точек Unicode) и задаёт единый учебный диапазон. В современном Python chr допускает и коды вне BMP; в лабораторной модели мы намеренно работаем внутри BMP, чтобы согласовать все формулы.

Принятые обозначения: M = 65536; для кодов открытого текста m и шифртекста c и ключа k:
- шифрование Цезаря: c = (m + k) mod M;
- дешифрование: m = (c − k) mod M.


## Ввод данных

Укажите фамилию, имя и группу (группа — по желанию). Эти данные используются в примерах и в заголовке итогового теста.


In [ ]:
from datetime import datetime

# Ввод данных от пользователя
student_name = input("Введите имя студента: ")
group_name   = input("Введите название группы: ")
sequence     = input("Введите порядковый номер: ")

# Вывод данных для заголовка теста
print("\nДанные для заголовка теста:")
print("- Имя:", student_name)
print("- Группа:", group_name)
print("- Порядковый номер:", sequence)

# Вывод текущего времени
now = datetime.now()
print("- Время:", now.strftime("%d.%m.%Y %H:%M:%S"))

In [ ]:
from collections import Counter
from dataclasses import dataclass
import secrets
import socket
from typing import Iterable, Optional, Tuple

UNICODE_MOD = 65536


def _wrap_u16(x: int) -> int:
    """Приводит целое число к диапазону [0, 65535] по модулю 65536."""
    return x % UNICODE_MOD


def _to_codes(s: str) -> list[int]:
    return [ord(ch) for ch in s]


def _from_codes(xs: Iterable[int]) -> str:
    return "".join(chr(_wrap_u16(x)) for x in xs)


def show_codes_preview(label: str, text: str, limit: int = 25) -> None:
    """Выводит подпись, строку и первые `limit` кодов символов (ord)."""
    codes = [ord(ch) for ch in text]
    head = codes[:limit]
    tail = (" ..." if len(codes) > limit else "")
    print(label)
    print("текст:", text)
    print("коды:", head, tail)


## Задание 1. Обобщённый шифр Цезаря

### Описание алгоритма
Обобщённый шифр Цезаря — это сдвиг каждого символа открытого текста на одно и то же целое число k (ключ). Вместо «алфавита» используются коды символов Unicode; сдвиг выполняется по модулю M = 65536.

Связь кодов: если m_i — код i-го символа открытого текста, c_i — код соответствующего символа шифртекста, то
c_i = (m_i + k) mod 65536,
m_i = (c_i − k) mod 65536.

Модуль 65536 задаёт учебную модель в диапазоне BMP и согласован с функциями _wrap_u16 и _from_codes.

### Требуется реализовать
- caesar_encrypt(k, m) — шифрование;
- caesar_decrypt(k, c) — дешифрование.

### Рекомендуемый порядок действий
1. Получить список кодов: codes = _to_codes(текст).
2. Применить формулу с приведением по модулю UNICODE_MOD.
3. Собрать строку: _from_codes(...).

### Проверка корректности
Для любого открытого текста m должно выполняться: caesar_decrypt(k, caesar_encrypt(k, m)) == m. Проверку выполните в следующей ячейке (без использования assert).


In [ ]:
def caesar_encrypt(k: int, m: str) -> str:
    """Шифрование строки m обобщённым шифром Цезаря.

    Параметры: k — целочисленный ключ (смещение); m — открытый текст.
    Возвращает шифртекст той же длины.
    """
    # TODO: реализуйте
    # codes = _to_codes(m)
    # shifted = [(x + k) % UNICODE_MOD for x in codes]
    # return _from_codes(shifted)
    raise NotImplementedError


def caesar_decrypt(k: int, c: str) -> str:
    """Дешифрование шифртекста c с ключом k."""
    # TODO: реализуйте
    # codes = _to_codes(c)
    # restored = [(x - k) % UNICODE_MOD for x in codes]
    # return _from_codes(restored)
    raise NotImplementedError


## Задание 1B. Шифр Виженера (после Цезаря)

### Идея
Шифр Виженера использует **строковый ключ**. Для каждого символа текста берётся символ ключа (ключ циклически повторяется), и выполняется сдвиг кода на величину `ord(символ_ключа)` по модулю `UNICODE_MOD`.

Для i-го символа:
- `c_i = (m_i + k_i) mod UNICODE_MOD`
- `m_i = (c_i - k_i) mod UNICODE_MOD`

где `k_i` — код i-го символа повторённого ключа.

### Почему это полезно
В отличие от Цезаря, сдвиг в Виженере меняется от позиции к позиции, поэтому простая частотная атака «одним сдвигом» уже не работает напрямую.


In [ ]:
def _repeat_key_to_length_vigenere(key: str, n: int) -> str:
    """Повторяет ключ до длины n (служебная функция для Виженера)."""
    if n < 0:
        raise ValueError("n must be non-negative")
    if n == 0:
        return ""
    if key == "":
        raise ValueError("key must be non-empty when n>0")
    q, r = divmod(n, len(key))
    return key * q + key[:r]


def vigenere_encrypt(key: str, plaintext: str) -> str:
    """Шифрование Виженера в учебной Unicode-модели (mod 65536)."""
    # TODO: реализуйте
    # if plaintext == "":
    #     return ""
    # k2 = _repeat_key_to_length_vigenere(key, len(plaintext))
    # out = []
    # for p, kk in zip(plaintext, k2):
    #     out.append(_wrap_u16(ord(p) + ord(kk)))
    # return _from_codes(out)
    raise NotImplementedError


def vigenere_decrypt(key: str, ciphertext: str) -> str:
    """Дешифрование Виженера в учебной Unicode-модели (mod 65536)."""
    # TODO: реализуйте
    # if ciphertext == "":
    #     return ""
    # k2 = _repeat_key_to_length_vigenere(key, len(ciphertext))
    # out = []
    # for c, kk in zip(ciphertext, k2):
    #     out.append(_wrap_u16(ord(c) - ord(kk)))
    # return _from_codes(out)
    raise NotImplementedError


In [ ]:
# Проверка Виженера (после реализации функций)

vig_plain = "Vigenere demo: проверка после Цезаря"
vig_key = "KEY"

try:
    vig_cipher = vigenere_encrypt(vig_key, vig_plain)
    vig_restored = vigenere_decrypt(vig_key, vig_cipher)

    print("Открытый текст:", vig_plain)
    print("Ключ:", vig_key)
    print("Фрагмент шифртекста:", repr(vig_cipher[:40]))
    print("Восстановлено:", vig_restored)
    print("Статус:", "OK" if vig_restored == vig_plain else "ОШИБКА")
except NotImplementedError:
    print("Сначала реализуйте vigenere_encrypt и vigenere_decrypt")


## Задание 2. Восстановление текста без ключа (частотная эвристика)

### Идея
Шифр Цезаря — моноалфавитная подстановка: один символ открытого текста всегда переходит в один и тот же символ шифртекста. Поэтому относительные частоты символов в типичном тексте сохраняются (с переименованием символов).

Для достаточно длинного текста на естественном языке чаще всего наибольшую частоту имеет пробел. Принимается эвристика: самый частый символ шифртекста соответствует пробелу в открытом тексте.

Шаги:
1. Найти символ с максимальной частотой в шифртексте.
2. Считать, что он получен из пробела.
3. Вычислить оценку ключа k.
4. Выполнить дешифрование с этим k.

### Вывод формулы для ключа
Если c = (m + k) mod 65536 и для пробела m = ord(' '), то k ≡ ord(c_частый) − ord(' ') (mod 65536).

### Ограничения метода
- Устойчивее на текстах объёмом порядка десятка слов и больше.
- На коротких строках лидирующий по частоте символ может быть не пробелом.
- При данных без типичной частотной структуры (например, случайные символы) эвристика неприменима.

### Дополнительно (по желанию)
Можно рассмотреть несколько наиболее частых символов шифртекста и выбрать вариант, дающий текстоподобный результат.


In [ ]:
@dataclass(frozen=True)
class CaesarCrackResult:
    """Результат эвристического взлома шифра Цезаря.

    Атрибуты: key — оценка ключа; plaintext — восстановленный текст;
    most_frequent_cipher_char — самый частый символ шифртекста.
    """

    key: int
    plaintext: str
    most_frequent_cipher_char: str


def caesar_crack_assuming_space(c: str) -> CaesarCrackResult:
    """Взлом по гипотезе: самый частый символ шифртекста — образ пробела."""
    # TODO: реализуйте
    # if c == "":
    #     return CaesarCrackResult(0, "", "")
    # top_char, _ = Counter(c).most_common(1)[0]
    # k = (ord(top_char) - ord(' ')) % UNICODE_MOD
    # plaintext = caesar_decrypt(k, c)
    # return CaesarCrackResult(k, plaintext, top_char)
    raise NotImplementedError


In [ ]:
# Демонстрация частотной эвристики (после реализации функций)

demo_plain = (
    "Это демонстрационный текст для частотного анализа. "
    "В нём достаточно пробелов, чтобы эвристика сработала устойчиво. "
    "Чем длиннее текст, тем надёжнее оценка."
)

demo_key = 1234

try:
    demo_cipher = caesar_encrypt(demo_key, demo_plain)
    result = caesar_crack_assuming_space(demo_cipher)

    print("Исходный ключ (mod 65536):", demo_key % 65536)
    print("Оценённый ключ (mod 65536):", result.key % 65536)
    print("Самый частый символ шифртекста:", repr(result.most_frequent_cipher_char))

    print("\nФрагмент восстановленного текста:")
    print(result.plaintext[:200])

    if result.plaintext == demo_plain:
        print("\nИтог: OK — открытый текст восстановлен")
    else:
        print("\nИтог: возможна ошибка эвристики (текст не совпал с исходным)")
except NotImplementedError:
    print("Сначала реализуйте caesar_encrypt, caesar_decrypt и caesar_crack_assuming_space")


## Задание 3. Шифр Вернама (операция XOR)

### Идея
В потоковых схемах часто используют поразрядное исключающее ИЛИ (XOR) между открытым текстом и ключевым потоком. В учебной записи шифртекст получается из открытого текста и ключа посредством XOR на уровне кодов символов.

Свойство: (x ⊕ k) ⊕ k = x, поэтому для XOR одна и та же операция применяется и при шифровании, и при дешифровании.

### Реализация в работе
XOR выполняется для целых кодов символов; результат приводится к диапазону [0, 65535] функцией _wrap_u16.

### Два режима ключа
- Повторяющийся ключ (аналог идеи шифра Виженера, но с XOR): строка-ключ короче сообщения и циклически повторяется.
- OTP (одноразовый блокнот): длина ключа не меньше длины сообщения; в классической модели биты ключа независимы и равновероятны, ключ используется один раз и хранится в секрете.

### Замечание о безопасности
Повтор ключа создаёт закономерности, удобные для криптоанализа. OTP при указанных условиях даёт теоретически стойкую схему; на практике трудно обеспечить длину и секретность ключа.


In [ ]:
def _repeat_key_to_length(key: str, n: int) -> str:
    """Повторяет строку-ключ до длины n.

    Пример: key='abc', n=8 -> 'abcabcab'
    """
    if n < 0:
        raise ValueError("n must be non-negative")
    if n == 0:
        return ""
    if key == "":
        raise ValueError("key must be non-empty when n>0")
    q, r = divmod(n, len(key))
    return key * q + key[:r]


def vernam_xor_encrypt_repeating_key(key: str, plaintext: str) -> str:
    """Шифрование открытого текста XOR с повторяющимся ключом.

    Параметры: key — непустая строка-ключ; plaintext — открытый текст.
    Возвращает шифртекст той же длины.
    """
    # TODO: реализуйте
    # if plaintext == "":
    #     return ""
    # k2 = _repeat_key_to_length(key, len(plaintext))
    # out = []
    # for p, kk in zip(plaintext, k2):
    #     out.append(_wrap_u16(ord(p) ^ ord(kk)))
    # return _from_codes(out)
    raise NotImplementedError


def vernam_xor_decrypt_repeating_key(key: str, ciphertext: str) -> str:
    """Дешифрование тем же XOR (операция совпадает с шифрованием)."""
    # TODO: реализуйте
    # return vernam_xor_encrypt_repeating_key(key, ciphertext)
    raise NotImplementedError


In [ ]:
def otp_generate_key(length: int) -> str:
    """Генерирует случайный ключ OTP заданной длины.

    Учебная модель: независимые равномерные значения в диапазоне [0, 65535].
    """
    if length < 0:
        raise ValueError("length must be non-negative")
    codes = [secrets.randbelow(UNICODE_MOD) for _ in range(length)]
    return _from_codes(codes)


def otp_encrypt(key: str, plaintext: str) -> str:
    """Шифрование в режиме OTP (XOR по кодам).

    Условие: len(key) == len(plaintext).
    """
    # TODO: реализуйте
    # if len(key) != len(plaintext):
    #     raise ValueError("OTP key must have same length as plaintext")
    # out = []
    # for p, k in zip(plaintext, key):
    #     out.append(_wrap_u16(ord(p) ^ ord(k)))
    # return _from_codes(out)
    raise NotImplementedError


def otp_decrypt(key: str, ciphertext: str) -> str:
    """Дешифрование OTP тем же XOR, что и шифрование."""
    # TODO: реализуйте
    # return otp_encrypt(key, ciphertext)
    raise NotImplementedError


In [ ]:
# Проверка: XOR с повтором ключа и OTP (после реализации функций)

m = f"Сообщение от {student_name or 'студента'}: секрет через XOR"
key = "ключ"

print("Открытый текст:")
print(m)

try:
    c = vernam_xor_encrypt_repeating_key(key, m)
    mm = vernam_xor_decrypt_repeating_key(key, c)

    print("\nКлюч (строка):", key)
    print("Шифртекст (внешний вид может быть необычным):")
    print(c)
    print("\nРасшифрованный текст:")
    print(mm)
    print("\nПовтор ключа — статус:", "OK" if mm == m else "ОШИБКА")
except NotImplementedError:
    print("\nСначала реализуйте vernam_xor_encrypt_repeating_key и vernam_xor_decrypt_repeating_key")

print("\n---\nОдноразовый блокнот (OTP)")
try:
    otp_k = otp_generate_key(len(m))
    c2 = otp_encrypt(otp_k, m)
    mm2 = otp_decrypt(otp_k, c2)
    print("Длина ключа OTP:", len(otp_k))
    print("OTP — статус:", "OK" if mm2 == m else "ОШИБКА")
except NotImplementedError:
    print("Сначала реализуйте otp_encrypt и otp_decrypt")


## Дополнительное задание A. Связь с шифром Виженера

Если реализован повторяющийся ключ и XOR, получается схема, близкая к идее шифра Виженера, но с XOR вместо сложения по модулю длины алфавита.

Вопросы для размышления:
- Почему при XOR фиксированной ширины (например, побайтово) результат остаётся в том же диапазоне и отдельное сложение по модулю после XOR не требуется?
- Почему для XOR операция дешифрования совпадает с операцией шифрования?


## Дополнительное задание B. Цепочка блоков (идея CBC) для XOR

### Назначение
Повторяющийся ключ приводит к повторяемым статистическим эффектам. Чтобы усложнить анализ, каждый блок открытого текста можно связать с предыдущим шифртекстом (идея режима CBC).

### Упрощённая модель в работе
- Блок — фрагмент строки длины len(key).
- IV — случайная строка той же длины, что и блок.
- Первый блок: сначала block XOR IV, затем результат XOR key.
- Следующие блоки: block XOR предыдущий_шифрблок, затем XOR key.

### Передача данных
Чтобы получатель мог расшифровать сообщение, IV обычно передают открыто вместе с шифртекстом. IV не является секретом, но в типичных режимах должен быть уникальным для данного ключа. В учебной реализации удобно вернуть пару (iv, ciphertext) или объединить их в одну строку.

### Подсказки
- Нужна функция XOR двух строк одинаковой длины.
- Если последний блок короче, применяется дополнение (padding); ниже — дополнение пробелами до длины, кратной длине ключа.


In [ ]:
from typing import Optional, Tuple

def xor_strings_same_length(a: str, b: str) -> str:
    """Побитовое XOR двух строк одинаковой длины по кодам символов."""
    if len(a) != len(b):
        raise ValueError("a and b must have same length")
    return _from_codes([_wrap_u16(ord(x) ^ ord(y)) for x, y in zip(a, b)])


def pad_to_block_size(s: str, block_size: int, pad_char: str = " ") -> str:
    """Дополняет строку до длины, кратной block_size, символом pad_char."""
    if block_size <= 0:
        raise ValueError("block_size must be > 0")
    if len(pad_char) != 1:
        raise ValueError("pad_char must be a single character")
    r = len(s) % block_size
    if r == 0:
        return s
    return s + pad_char * (block_size - r)


def cbc_xor_encrypt(key: str, plaintext: str, iv: Optional[str] = None) -> Tuple[str, str, int]:
    """CBC-подобное шифрование поверх XOR (учебная модель).

    Возвращает: iv, шифртекст, исходную длину открытого текста (для снятия padding).
    """
    if key == "":
        raise ValueError("key must be non-empty")
    block_size = len(key)
    original_len = len(plaintext)
    pt = pad_to_block_size(plaintext, block_size)
    if iv is None:
        iv = otp_generate_key(block_size)
    if len(iv) != block_size:
        raise ValueError("iv must have length == len(key)")
    out_blocks, prev = [], iv
    for i in range(0, len(pt), block_size):
        block = pt[i:i+block_size]
        cipher_block = xor_strings_same_length(xor_strings_same_length(block, prev), key)
        out_blocks.append(cipher_block)
        prev = cipher_block
    return iv, "".join(out_blocks), original_len


def cbc_xor_decrypt(key: str, iv: str, ciphertext: str, original_len: int) -> str:
    """CBC-подобное дешифрование (учебная модель)."""
    if key == "":
        raise ValueError("key must be non-empty")
    block_size = len(key)
    if len(iv) != block_size:
        raise ValueError("iv must have length == len(key)")
    if len(ciphertext) % block_size != 0:
        raise ValueError("ciphertext length must be multiple of block_size")
    out_blocks, prev = [], iv
    for i in range(0, len(ciphertext), block_size):
        cipher_block = ciphertext[i:i+block_size]
        block = xor_strings_same_length(xor_strings_same_length(cipher_block, key), prev)
        out_blocks.append(block)
        prev = cipher_block
    return "".join(out_blocks)[:original_len]

## Практика: обмен сообщениями по TCP (сервер и клиент «эхо»)

### Цель
Применить реализованные функции шифрования и дешифрования при передаче данных по сети.

### Где здесь TLS/SSL
- В этой лабораторной вы реализуете **учебную модель** защищённого обмена вручную.
- В реальных системах это обычно делает **TLS**: рукопожатие, проверка сертификата и защищённый канал.


### Протокол TCP (кратко)
- TCP передаёт поток байтов; чтобы выделять сообщения, стороны договариваются о правилах фрагментации.
- В задании используется соглашение: одна строка текста — одно сообщение; конец строки — символ перевода строки.

### Кодирование шифртекста для передачи
Шифртекст может содержать произвольные символы; передавать его «как есть» в текстовом виде неудобно. Поэтому строка переводится в последовательность десятичных кодов ord(...), коды записываются через пробел, строка завершается переводом строки. На приёме выполняется обратное преобразование через chr. Это транспортное кодирование; криптографическое преобразование задаётся вашими функциями шифрования и дешифрования.

### Отладка


In [ ]:
def pack_text_as_decimal_codes(s: str) -> str:
    """Упаковка строки: коды символов в десятичном виде через пробел."""
    return " ".join(str(ord(ch)) for ch in s)


def unpack_decimal_codes_to_text(s: str) -> str:
    """Восстановление строки из формата «числа через пробел»."""
    s = s.strip()
    if s == "":
        return ""
    nums = [int(x) for x in s.split()]
    return "".join(chr(_wrap_u16(n)) for n in nums)


def send_line(sock: socket.socket, line: str) -> None:
    data = (line + "\n").encode("utf-8")
    sock.sendall(data)


def recv_line(sock: socket.socket) -> str:
    buf = bytearray()
    while True:
        chunk = sock.recv(1)
        if chunk == b"":
            raise ConnectionError("socket closed")
        if chunk == b"\n":
            break
        buf.extend(chunk)
    return buf.decode("utf-8")


### Задание TCP-1: сервер

Реализуйте сервер «эхо», который:
1) принимает от клиента упакованный шифртекст (строка целых кодов);
2) распаковывает и дешифрует сообщение, выводит открытый текст;
3) формирует ответ в виде ОТВЕТ: <сообщение>;
4) шифрует ответ и отправляет его клиенту.

Подключите ваши функции шифрования и дешифрования вместо пометок TODO.


In [ ]:
HOST = "127.0.0.1"
PORT = 9009


def encrypt_for_transport(plaintext: str) -> str:
    """Шифрование и упаковка для передачи по TCP (реализуйте)."""
    # TODO: выберите один алгоритм и вызовите свою функцию.
    # Пример: Цезарь
    # k = 1234
    # ciphertext = caesar_encrypt(k, plaintext)
    # return pack_text_as_decimal_codes(ciphertext)
    # Пример: XOR с повтором
    # key = "ключ"
    # ciphertext = vernam_xor_encrypt_repeating_key(key, plaintext)
    # return pack_text_as_decimal_codes(ciphertext)
    raise NotImplementedError


def decrypt_from_transport(packed_ciphertext: str) -> str:
    """Распаковка и дешифрование (тот же алгоритм и ключ, что в encrypt_for_transport)."""
    ciphertext = unpack_decimal_codes_to_text(packed_ciphertext)
    # TODO: используйте тот же алгоритм и ключ, что и в encrypt_for_transport
    # k = 1234
    # return caesar_decrypt(k, ciphertext)
    # key = "ключ"
    # return vernam_xor_decrypt_repeating_key(key, ciphertext)
    raise NotImplementedError


def run_echo_server() -> None:
    """Однократное принятие соединения: приём — дешифрование — ответ — шифрование."""
    print(f"Ожидание подключения на {HOST}:{PORT} ...")
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server:
        server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        server.bind((HOST, PORT))
        server.listen(1)

        conn, addr = server.accept()
        with conn:
            print("Подключён клиент:", addr)

            packed = recv_line(conn)
            msg = decrypt_from_transport(packed)
            print("Расшифрованное сообщение от клиента:", msg)

            reply_plain = "ОТВЕТ: " + msg
            packed_reply = encrypt_for_transport(reply_plain)
            send_line(conn, packed_reply)


# Запуск сервера — в отдельном процессе или терминале:
# run_echo_server()


### Задание TCP-2: клиент

Клиент должен:
1) взять открытый текст;
2) зашифровать и упаковать его;
3) отправить на сервер;
4) принять ответ, распаковать и расшифровать.

На клиенте и сервере должны совпадать алгоритм и ключ.


In [ ]:
def run_client(message: str) -> str:
    """Клиент: шифрование — отправка — приём — дешифрование."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.connect((HOST, PORT))

        packed = encrypt_for_transport(message)
        send_line(sock, packed)

        packed_reply = recv_line(sock)
        return decrypt_from_transport(packed_reply)


# Пример:
# 1) в другом процессе: run_echo_server()
# 2) здесь: print(run_client("Привет, сервер"))


## Контрольные вопросы (письменные ответы)

1) В чём состоит алгоритм частотного анализа? Что измеряют? Почему метод применим к моноалфавитной подстановке?

2) Какие распространённые классы атак на криптографические примитивы вам известны?
- полный перебор ключа;
- атака по известному открытому тексту;
- атака по выбранному открытому тексту;
- побочные каналы (например, по времени или энергопотреблению).

3) Какие симметричные шифры широко применяются сегодня и считаются стойкими при правильном использовании?
- AES (режимы вроде GCM, CTR и др.);
- ChaCha20-Poly1305.

Отдельно: почему одного лишь AES-CBC без аутентификации недостаточно для многих прикладных задач (роль AEAD или отдельного MAC).


## Отчёт по работе
- Исходный код всех реализованных функций.
- Скриншоты или вывод проверочных ячеек с пометками OK / ОШИБКА.
- Краткое описание эксперимента: при каких текстах частотный взлом Цезаря дал верный результат и при каких — нет, и почему.
- Комментарий по части TCP: какой алгоритм выбран для транспорта и почему.


## Итоговый тест: 20 вопросов по симметричному шифрованию

Инструкция:
- выберите вариант, нажав кнопку;
- верный ответ подсвечивается зелёным;
- при ошибке выбранный вариант подсвечивается красным, верный — зелёным;
- вверху отображаются имя, группа (если указаны) и текущий счёт.


In [ ]:
# In terminal

pip install --upgrade pip

pip install ipywidgets


In [ ]:
# Интерактивный тест (20 вопросов)
# Нужен пакет ipywidgets (обычно доступен в JupyterLab)

import ipywidgets as widgets
from IPython.display import display, HTML

questions = [
    {
        "q": "1) Что характеризует симметричное шифрование?",
        "options": [
            "Один и тот же ключ используется для шифрования и расшифрования",
            "Всегда используются два разных ключа",
            "Ключ хранится только у сервера",
            "Секретность не зависит от ключа",
        ],
        "answer": 0,
    },
    {
        "q": "2) Основная слабость моноалфавитной подстановки (например, Цезарь):",
        "options": [
            "Нельзя расшифровать даже с ключом",
            "Не работает с цифрами",
            "Сохраняется частотная структура текста",
            "Слишком большой ключ",
        ],
        "answer": 2,
    },
    {
        "q": "3) Для обобщённого Цезаря в этой лабораторной используется модуль:",
        "options": ["26", "255", "65536", "1114112"],
        "answer": 2,
    },
    {
        "q": "4) Что возвращает функция ord(c) в Python?",
        "options": [
            "Строку из одного символа",
            "Целочисленный код символа",
            "Бинарный ключ",
            "Хэш символа",
        ],
        "answer": 1,
    },
    {
        "q": "5) Какое свойство XOR позволяет использовать одну и ту же операцию для шифрования и дешифрования?",
        "options": [
            "Коммутативность (перестановка слагаемых)",
            "Инволютивность: (x XOR k) XOR k = x",
            "XOR всегда увеличивает число",
            "XOR требует модуль",
        ],
        "answer": 1,
    },
    {
        "q": "6) OTP (одноразовый блокнот) теоретически стойкий, если:",
        "options": [
            "Ключ короче сообщения и повторяется",
            "Ключ такой же длины, что и сообщение; биты ключа независимы и равновероятны; ключ используется один раз и хранится в секрете",
            "Ключ хранится в открытом виде",
            "Используется только латиница",
        ],
        "answer": 1,
    },
    {
        "q": "7) Почему повторяющийся ключ в XOR-схеме опасен?",
        "options": [
            "Появляются статистические и периодические зависимости",
            "Нельзя выполнить дешифрование",
            "Ключ становится длиннее сообщения",
            "Невозможно передать данные по сети",
        ],
        "answer": 0,
    },
    {
        "q": "8) Что обычно передают вместе с шифртекстом в CBC-подобных схемах?",
        "options": ["Секретный ключ шифрования", "IV (вектор инициализации)", "Пароль пользователя", "Хэш пароля"],
        "answer": 1,
    },
    {
        "q": "9) Какова роль IV?",
        "options": [
            "Заменяет ключ",
            "Гарантирует, что один и тот же открытый текст всегда даст один и тот же шифртекст при том же ключе",
            "Снижает повторяемость паттернов между сообщениями",
            "Ускоряет дешифрование за счёт удаления ключа",
        ],
        "answer": 2,
    },
    {
        "q": "10) Частотный анализ обычно работает лучше всего, когда:",
        "options": [
            "Текст достаточно длинный",
            "Текст состоит из 2–3 символов",
            "Ключ очень маленький",
            "Используется только ASCII",
        ],
        "answer": 0,
    },
    {
        "q": "11) Какой из перечисленных вариантов относится к AEAD (аутентифицированному шифрованию)?",
        "options": ["AES-ECB", "AES-CBC без аутентификации (без MAC/AEAD)", "AES-GCM", "ROT13"],
        "answer": 2,
    },
    {
        "q": "12) Почему AES-ECB считается небезопасным для многих задач?",
        "options": [
            "Не поддерживает ключи",
            "Одинаковые блоки открытого текста дают одинаковые блоки шифртекста",
            "Слишком медленный",
            "Требует интернет",
        ],
        "answer": 1,
    },
    {
        "q": "13) Что обязательно для корректного OTP в этой лабораторной?",
        "options": [
            "len(key) == len(message)",
            "len(key) < len(message)",
            "Ключ должен быть строкой 'ключ'",
            "Ключ должен состоять только из цифр",
        ],
        "answer": 0,
    },
    {
        "q": "14) В модели CBC из лабораторной размер блока равен:",
        "options": [
            "1 байт",
            "len(key)",
            "len(message)",
            "всегда 128 бит",
        ],
        "answer": 1,
    },
    {
        "q": "15) Что верно про симметричные алгоритмы в целом?",
        "options": [
            "Они всегда медленнее асимметричных",
            "Они обычно быстрее асимметричных и подходят для шифрования данных",
            "Они не используют ключи",
            "Они подходят только для цифровых подписей",
        ],
        "answer": 1,
    },
    {
        "q": "16) Что делает функция chr(i) в Python?",
        "options": [
            "Возвращает код символа",
            "Возвращает символ по числовому коду",
            "Шифрует строку",
            "Делает XOR",
        ],
        "answer": 1,
    },
    {
        "q": "17) Что критично при TCP-обмене зашифрованными сообщениями?",
        "options": [
            "Игнорировать границы сообщений",
            "Задать формат границ сообщений (например, одна строка — одно сообщение, конец — перевод строки)",
            "Передавать только латинские символы",
            "Не декодировать байты",
        ],
        "answer": 1,
    },
    {
        "q": "18) Если дешифрование не восстанавливает исходный текст, сначала стоит проверить:",
        "options": [
            "Одинаковый ли ключ/алгоритм используется на обеих сторонах",
            "Размер экрана в Jupyter",
            "Скорость интернета",
            "Цвет темы IDE",
        ],
        "answer": 0,
    },
    {
        "q": "19) Какой современный вариант потокового шифрования + аутентификации широко используется?",
        "options": ["ChaCha20-Poly1305", "Шифр Виженера", "Шифр Цезаря", "DES в режиме ECB"],
        "answer": 0,
    },
    {
        "q": "20) Главное педагогическое ограничение этой лабораторной:",
        "options": [
            "Рассматриваются учебные модели, а не промышленные криптосистемы",
            "Нельзя использовать Python",
            "Нельзя запускать код",
            "Все алгоритмы абсолютно стойкие",
        ],
        "answer": 0,
    },
]

score = {"correct": 0, "answered": 0}
question_states = [False] * len(questions)

student_label = student_name if str(student_name).strip() else "Студент"
group_label = group_name if str(group_name).strip() else "(группа не указана)"

header = widgets.HTML("<h3 style='margin:0;'>Тест по симметричному шифрованию</h3>")
progress = widgets.HTML("")


def update_progress():
    total = len(questions)
    correct = score["correct"]
    answered = score["answered"]
    percent = (100.0 * correct / total) if total else 0.0

    status = ""
    if answered == total:
        status = "<span style='color:#0a7f2e;'><b>Тест завершён.</b></span>"

    progress.value = (
        f"<b>Студент:</b> {student_label} | <b>Группа:</b> {group_label}<br>"
        f"<b>Результат:</b> {correct} / {total} ({percent:.1f}%) | "
        f"<b>Отвечено:</b> {answered} / {total}<br>{status}"
    )


def make_question_widget(q_idx, item):
    title = widgets.HTML(f"<b>{item['q']}</b>")
    feedback = widgets.HTML("<span style='color:#555;'>Выберите один вариант.</span>")

    buttons = []
    box = widgets.VBox()

    def on_click_factory(opt_idx):
        def _handler(btn):
            if question_states[q_idx]:
                return

            question_states[q_idx] = True
            score["answered"] += 1

            correct_idx = item["answer"]
            for i, b in enumerate(buttons):
                b.disabled = True
                if i == correct_idx:
                    b.button_style = "success"
                if i == opt_idx and opt_idx != correct_idx:
                    b.button_style = "danger"

            if opt_idx == correct_idx:
                score["correct"] += 1
                feedback.value = "<span style='color:green;'><b>Верно.</b></span>"
            else:
                correct_text = item["options"][correct_idx]
                feedback.value = (
                    "<span style='color:red;'><b>Неверно.</b></span> "
                    f"<span style='color:#333;'>Правильный ответ: {correct_text}</span>"
                )

            update_progress()
        return _handler

    for i, opt in enumerate(item["options"]):
        b = widgets.Button(
            description=opt,
            button_style="",
            layout=widgets.Layout(width="95%"),
            style={"button_color": None},
        )
        b.on_click(on_click_factory(i))
        buttons.append(b)

    box.children = [title] + buttons + [feedback, widgets.HTML("<hr style='margin:10px 0;'>")]
    return box


all_questions = [make_question_widget(i, q) for i, q in enumerate(questions)]
quiz_ui = widgets.VBox([header, progress] + all_questions)

update_progress()
display(quiz_ui)
